-- 1. 创建学生选课数据库
CREATE DATABASE IF NOT EXISTS db_experiment_week03
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci;

-- 2. 使用该数据库
USE db_experiment_week03;

-- 3. 创建 Student 表（学生表）
CREATE TABLE Student (
    ID VARCHAR(20) NOT NULL COMMENT '学号',
    Name VARCHAR(50) NOT NULL COMMENT '姓名',
    Sex ENUM('男', '女') NOT NULL COMMENT '性别',
    Age INT COMMENT '年龄',
    Dept VARCHAR(50) NOT NULL COMMENT '系别',
    PRIMARY KEY (ID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='学生表';

-- 4. 插入学生数据（包含课件中使用的CS系学生）
INSERT INTO Student (ID, Name, Sex, Age, Dept) VALUES
('200215121', '李勇', '男', 20, 'CS'),
('200215122', '刘晨', '女', 19, 'CS'),
('200215123', '王敏', '女', 18, 'MA'),
('200215124', '陈立', '男', 21, 'CS'),
('200215125', '张立', '男', 19, 'IS'),
('200215126', '赵华', '女', 20, 'CS'),
('200215127', '孙强', '男', 20, 'EE');   -- 没有选课，用于演示 LEFT JOIN

-- 5. 创建 Course 表（课程表）
CREATE TABLE IF NOT EXISTS Course (
    CourseID VARCHAR(10) NOT NULL COMMENT '课程编号',
    CourseName VARCHAR(100) NOT NULL COMMENT '课程名称',
    Credit INT COMMENT '学分',
    PRIMARY KEY (CourseID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='课程表';

-- 6. 插入课程数据
INSERT INTO Course (CourseID, CourseName, Credit) VALUES
('C001', '数据库系统概论', 4),
('C002', '数据结构', 3),
('C003', '高等数学', 5),
('C004', '计算机网络', 3),
('C005', '操作系统', 4),
('C006', '人工智能导论', 2);   -- 无人选课，用于演示 RIGHT JOIN

-- 7. 创建 SC 表（选课表）- 对应课件中的SC
CREATE TABLE IF NOT EXISTS SC (
    StudentID VARCHAR(20) NOT NULL COMMENT '学号',
    CourseID VARCHAR(10) NOT NULL COMMENT '课程编号',
    Grade INT COMMENT '成绩',
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Student(ID) ON DELETE CASCADE,
    FOREIGN KEY (CourseID) REFERENCES Course(CourseID) ON DELETE CASCADE
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='选课表';

-- 8. 插入选课数据（包含Grade > 90的成绩，用于课件示例）
INSERT INTO SC (StudentID, CourseID, Grade) VALUES
('200215121', 'C001', 95),
('200215121', 'C002', 88),
('200215121', 'C003', 92),
('200215122', 'C001', 98),
('200215122', 'C004', 85),
('200215123', 'C002', 78),
('200215123', 'C005', 82),
('200215124', 'C001', 91),
('200215124', 'C003', 87),
('200215125', 'C001', 76),
('200215125', 'C004', 94),
('200215126', 'C002', 96),
('200215126', 'C005', 89);

# 数据库实验课件

# 关系代数与SQL查询（40页PPT结构）

---

# 第一部分 关系代数基础（6页）

## 第1页 课程目标

理解关系代数：

* 选择
* 投影
* 连接
* 并
* 交
* 差
* 除

能够：

```
关系代数 → SQL
SQL → 关系代数
```

---

## 第2页 实验数据库结构

数据库：

```
db_experiment_week03
```

表：

```
Student(ID, Name, Sex, Age, Dept)

Course(CourseID, CourseName, Credit)

SC(StudentID, CourseID, Grade)
```

关系图：

```
Student 1 ----- n SC n ----- 1 Course
```

---

## 第3页 关系代数运算分类

关系代数运算分为：

### 基本运算

| 运算   | 符号 |
| ---- | -- |
| 选择   | σ  |
| 投影   | π  |
| 笛卡尔积 | ×  |
| 连接   | ⨝  |

### 集合运算

| 运算 | 符号 |
| -- | -- |
| 并  | ∪  |
| 交  | ∩  |
| 差  | −  |

### 特殊运算

```
除 ÷
```

---

## 第4页 关系代数执行流程

查询过程：

```
关系代数表达式
        ↓
查询树
        ↓
SQL查询
        ↓
数据库执行
```

---

# 第二部分 基本关系代数（10页）

---

# 第5页 选择（Selection）

选择：

```
σ 条件 (R)
```

例：

```
σ Dept='CS'(Student)
```

含义：

```
查询CS系学生
```

SQL：

```sql
SELECT *
FROM Student
WHERE Dept='CS';
```

---

# 第6页 投影（Projection）

投影：

```
π 属性 (R)
```

例：

```
π Name,Dept(Student)
```

SQL：

```sql
SELECT Name,Dept
FROM Student;
```

---

# 第7页 笛卡尔积

关系代数：

```
Student × Course
```

含义：

```
所有学生 × 所有课程
```

SQL：

```sql
SELECT *
FROM Student, Course;
```

---

# 第8页 连接（Join）
在关系代数中，连接（Join）是通过两个或多个关系的公共属性将元组组合起来的操作。根据连接条件的不同，主要分为以下几种类型：

### 1. 基本连接类型

#### （1）θ连接（Theta Join）
-   **定义**：从两个关系的笛卡尔积中选取满足特定条件（θ）的元组。
-   **表示**：

$
R \bowtie_{R.A = S.B} S = \sigma_{R.A = S.B}(R \times S)
$

- **解释**：
  - $( R \times S )$：计算两个关系的笛卡尔积。
  - $( \sigma )$（选择操作符）：筛选出满足 $( R.A = S.B )$ 条件的元组。
-   **说明**：θ可以是比较运算符（如 $( >, <, =, \ge, \le, \ne )$）。当θ为“=”时，就是下面要介绍的自然连接或等值连接。

#### （2）等值连接（Equi-Join）
-   **定义**：θ连接的一种特例，连接条件为“属性相等”。
-   **表示**：$( R \bowtie_{R.A = S.B} S )$
-   **特点**：结果集中会包含两个关系的重复属性（即两个用于比较的属性列都会保留）。

#### （3）自然连接（Natural Join）
-   **定义**：一种特殊的等值连接，要求两个关系**必须有相同的属性名**，并且在结果中**去掉重复的属性列**。
-   **表示**：$( R \bowtie S )$ 或 $( R * S )$
-   **特点**：自动比较两个关系中所有同名的属性，要求值相等，并合并为一列。

---

# 连接：

```
Student ⨝ SC
```

关系代数：

```
Student ⨝ Student.ID=SC.StudentID SC
```

SQL：

```sql
SELECT *
FROM Student
JOIN SC
ON Student.ID = SC.StudentID;
```

---

# 第9页 三表连接

关系代数：

```
(Student ⨝ SC) ⨝ Course
```

SQL：

```sql
SELECT *
FROM Student
JOIN SC ON Student.ID = SC.StudentID
JOIN Course ON SC.CourseID = Course.CourseID;
```

---
很好，这一页已经是**内连接（INNER JOIN）**的基础形式了 👍
我帮你在课件中**系统扩展四种连接类型**，保持和你现有风格一致，可直接放进PPT👇

---

# 内连接（INNER JOIN）

## 含义

只保留**两个表中匹配成功的记录**

## 关系代数

$ R ;\bowtie_{\theta}; S $
$
\sigma_{\theta}(R \times S)
$

## 含义

* 先做笛卡尔积 $(R \times S)$
* 再按条件 $(\theta)$ 进行选择

## SQL

```sql
SELECT *
FROM Student
INNER JOIN SC
ON Student.ID = SC.StudentID;
```

## 特点

* 只返回“有选课记录的学生”
* 没有匹配的行会被丢弃

---

# 左连接（LEFT JOIN）

## 数学表达式
# R⟕S

## 等价展开

$
(R \bowtie_{\theta} S)
\;\cup\;
\Big(
(R - \pi_{R}(R \bowtie_{\theta} S))
\times \text{NULL}_S
\Big)
$

## 含义

* 包含所有内连接结果
* 加上“在 R 中但未匹配的元组”，右侧补 NULL

---

## SQL

```sql
SELECT *
FROM Student
LEFT JOIN SC
ON Student.ID = SC.StudentID;
```

## 特点

* 所有学生都会显示
* 没选课的学生 → SC字段为 NULL

---

# 右连接（RIGHT JOIN）

## 数学表达式
# R⟖S

## 等价展开

$
(R \bowtie_{\theta} S)
;\cup;
\Big( (S - \pi_{S}(R \bowtie_{\theta} S)) \times {NULL_R} \Big)
$

## 含义

* 包含所有内连接结果
* 加上“在 S 中但未匹配的元组”，左侧补 NULL

---
## SQL

```sql
SELECT *
FROM SC
RIGHT JOIN Course
ON SC.CourseID = Course.CourseID;
```

## 特点

* 所有选课记录都会显示
* 找不到学生的信息 → Student字段为 NULL

---

# 全外连接（FULL OUTER JOIN）

## 数学表达式

# R⟗S

## 等价展开

$
(R \bowtie_{\theta} S)
;\cup;
\Big( (R - \pi_{R}(R \bowtie_{\theta} S)) \times {NULL_S} \Big)
;\cup;
\Big( (S - \pi_{S}(R \bowtie_{\theta} S)) \times {NULL_R} \Big)
$

---

## SQL

```sql
SELECT *
FROM Student
FULL OUTER JOIN SC
ON Student.ID = SC.StudentID;
```
## MySQL SQL 语法 
-- 左外连接部分
SELECT *
FROM Student
LEFT JOIN SC ON Student.ID = SC.StudentID

UNION

-- 右外连接部分
SELECT *
FROM Student
RIGHT JOIN SC ON Student.ID = SC.StudentID;

or

SELECT *
FROM SC
LEFT JOIN Course ON SC.CourseID = Course.CourseID

UNION

SELECT *
FROM SC
RIGHT JOIN Course ON SC.CourseID = Course.CourseID;
## 特点

* 左右表全部保留
* 不匹配部分用 NULL 填充

⚠️ 注意：

* MySQL 不支持 FULL OUTER JOIN（需用 UNION 实现）

---

# 连接类型对比总结

| 类型         | 是否保留左表 | 是否保留右表 | 未匹配处理   |
| ---------- | ------ | ------ | ------- |
| INNER JOIN | ❌      | ❌      | 丢弃      |
| LEFT JOIN  | ✅      | ❌      | 右表 NULL |
| RIGHT JOIN | ❌      | ✅      | 左表 NULL |
| FULL OUTER | ✅      | ✅      | 双方 NULL |

---
内连接：求两个关系的交集（匹配的部分）

左外连接：左关系全部 + 右关系的匹配部分

右外连接：右关系全部 + 左关系的匹配部分

全外连接：两个关系的并集（全部保留，NULL 填充缺失）

# 第15页 教学建议（可选）

可以加一个提问页：

👉 哪种连接用于：

* 查询“所有学生（包括未选课）”？
  → **LEFT JOIN**

👉 查询“只选课的学生”？
→ **INNER JOIN**

👉 查询“所有课程（即使没人选）”？
→ **RIGHT JOIN（或Course LEFT JOIN SC）**

---

# 第三部分 集合运算（12页）

---

# 第10页 并运算

关系代数：

```
σ Dept='CS'(Student)
∪
σ Dept='IS'(Student)
```

SQL：

```sql
SELECT *
FROM Student
WHERE Dept='CS'

UNION

SELECT *
FROM Student
WHERE Dept='IS';
```

---

# 第11页 并运算图解

```
CS学生 ∪ IS学生
```

集合示意：

```
A ∪ B
```

---

# 第12页 交运算

关系代数：

```
π StudentID (σ CourseID='C001'(SC))
∩
π StudentID (σ CourseID='C003'(SC))
```

SQL：

```sql
SELECT DISTINCT A.StudentID
FROM SC A
JOIN SC B
ON A.StudentID=B.StudentID
WHERE A.CourseID='C001'
AND B.CourseID='C003';
```

---

# 第13页 交运算图

```
A ∩ B
```

含义：

```
同时满足条件
```

---

# 第14页 差运算

关系代数：

```
π StudentID (σ CourseID='C001'(SC))
−
π StudentID (σ CourseID='C003'(SC))
```

SQL：

```sql
SELECT StudentID
FROM SC
WHERE CourseID='C001'
AND StudentID NOT IN (
        SELECT StudentID
        FROM SC
        WHERE CourseID='C003'
        );
```

---

# 第15页 差运算图

```
A − B
```

含义：

```
只在A中
```

---

# 第四部分 除法运算（8页）

---

# 第16页 除法问题

查询：

```
选修所有课程的学生
```

---

# 第17页 关系代数

关系：

```
R(StudentID,CourseID)

S(CourseID)
```

除法：

```
R ÷ S
```

含义：

```
选修了S中所有课程的学生
```

---

# 第18页 除法逻辑

逻辑：

```
不存在一门课程
该学生没有选
```

---

# 第19页 SQL实现

```sql
SELECT DISTINCT S1.StudentID
FROM SC S1
WHERE NOT EXISTS
(
SELECT *
FROM Course C
WHERE NOT EXISTS
(
SELECT *
FROM SC S2
WHERE S2.StudentID = S1.StudentID
AND S2.CourseID = C.CourseID
)
);
```

---

# 第20页 除法图

```
SC

Student Course

A      C1
A      C2
B      C1
B      C2
C      C1

Course

C1
C2
```

结果：

```
A
B
```

---

# 第五部分 综合查询（6页）

---

# 第21页 CS系学生选课

关系代数：

```
π Name,CourseName
(
σ Dept='CS'(Student)
⨝
SC
⨝
Course
)
```

SQL：

```sql
SELECT S.Name, C.CourseName
FROM Student S
JOIN SC ON S.ID = SC.StudentID
JOIN Course C ON SC.CourseID = C.CourseID
WHERE S.Dept='CS';
```

---

# 第22页 成绩90以上学生

关系代数：

```
σ Grade>90(SC)
```

SQL：

```sql
SELECT *
FROM SC
WHERE Grade>90;
```

---

# 第六部分 实验练习（8页）

---

# 第23页 练习1

关系代数：

```
π Name(Student)
```

SQL：

```sql
SELECT Name
FROM Student;
```

---

# 第24页 练习2

关系代数：

```
σ Grade>85(SC)
```

SQL：

```sql
SELECT *
FROM SC
WHERE Grade>85;
```

---

# 第25页 练习3

关系代数：

```
π StudentID (σ CourseID='C001'(SC))
```

SQL：

```sql
SELECT StudentID
FROM SC
WHERE CourseID='C001';
```

---

# 第七部分 总结（4页）

---

# 第26页 运算总结

| 运算 | 关系代数 | SQL        |
| -- | ---- | ---------- |
| 选择 | σ    | WHERE      |
| 投影 | π    | SELECT     |
| 连接 | ⨝    | JOIN       |
| 并  | ∪    | UNION      |
| 差  | −    | NOT IN     |
| 交  | ∩    | JOIN       |
| 除  | ÷    | NOT EXISTS |

---

# 第27页 关系代数口诀

```
选择 σ
投影 π
连接 ⨝
并 ∪
交 ∩
差 −
除 ÷
```

---

# 第28页 SQL口诀

```
SELECT
FROM
JOIN
WHERE
GROUP BY
HAVING
ORDER BY
```

---

# 第29页 课程总结

学生需要掌握：

```
关系代数表达式
SQL查询
关系模型
```

---

# 第30页 实验提交

提交：

```
SQL文件
实验报告
查询结果截图
```

---